# System Preparation for the SPINK7-KLK5 Pipeline

This notebook walks through the preparation stage: structure acquisition, chain selection, protonation-state assignment, topology construction, and explicit-solvent setup. The goal is to produce a chemically consistent OpenMM-ready system while preserving the physical assumptions ratified in the project blueprint.

At this stage, the most important biophysical choices are the protonation state, the force field, and the solvent model. In the current pipeline, the defaults are AMBER protein parameters with TIP3P water and physiological ionic strength.

## Preparation Rationale

The preparation stack enforces a specific thermodynamic and structural contract before any dynamics are launched. Protonation matters because the electrostatic energy depends directly on the charge state of titratable residues. Solvation matters because the Hamiltonian of the solvated system differs qualitatively from the vacuum Hamiltonian.

A useful thermodynamic reference is the thermal energy scale

$$k_B T = 0.008314462618Tathrm{kJmol^{-1}}$$

which sets the scale for fluctuations and helps interpret whether a structural perturbation is negligible or thermodynamically meaningful.

In [ ]:
# Resolve the project root and add it to the path for src imports.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import pipeline modules for structure retrieval, cleaning, and solvation.
from src.config import DATA_DIR, PROJECT_ROOT as CONFIG_PROJECT_ROOT, SystemConfig
from src.prep.pdb_fetch import fetch_alphafold, fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.protonate import assign_protonation
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system

system_config = SystemConfig()
print('Notebook project root:', PROJECT_ROOT)
print('Configured data directory:', DATA_DIR)
print(system_config)

## Choose a Structure Source

Use one of the two paths below:

1. Fetch an experimental RCSB structure with `fetch_pdb`.
2. Fetch an AlphaFold model with `fetch_alphafold`.

For a portfolio walkthrough, it is often useful to start from a single explicit example and keep the rest of the notebook parameterized.

In [ ]:
raw_dir = DATA_DIR / 'pdb' / 'raw'
prepared_dir = DATA_DIR / 'pdb' / 'prepared'
raw_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)

# Example inputs. Uncomment one of the retrieval lines when running interactively.
pdb_id = '2PSX'
uniprot_id = 'Q6UWN8'

# raw_structure_path = fetch_pdb(pdb_id, raw_dir)
# raw_structure_path = fetch_alphafold(uniprot_id, raw_dir)
raw_structure_path = raw_dir / f'{pdb_id}.pdb'
raw_structure_path

In [ ]:
# Run the full preparation pipeline: clean -> protonate -> build topology -> solvate.
chains_to_keep = ['A', 'B']

cleaned_path = clean_structure(raw_structure_path, chains_to_keep=chains_to_keep)
protonated_path = assign_protonation(cleaned_path, ph=system_config.ph, force_field='AMBER')
topology, vacuum_system, modeller = build_topology(protonated_path, system_config)
solvated_modeller, n_water, n_positive_ions, n_negative_ions = solvate_system(modeller, system_config)

summary = {
    'cleaned_path': cleaned_path,
    'protonated_path': protonated_path,
    'n_atoms_before_solvent': topology.getNumAtoms(),
    'n_atoms_after_solvent': solvated_modeller.topology.getNumAtoms(),
    'n_water': n_water,
    'n_positive_ions': n_positive_ions,
    'n_negative_ions': n_negative_ions,
}
summary

## Interpreting the Output

A successful preparation run should leave you with a protonated PDB, a modeller containing periodic box vectors, and solvent/ion counts that are physically plausible for the requested padding and ionic strength. Before moving on to equilibration, inspect the protonation log and verify that the selected chains match the intended complex definition.